In [2]:
# Email Spam Detection using Machine Learning and Generative AI

## Objective

# The objective of this project is to detect whether an email is
# Spam or Not Spam using Machine Learning.

# Generative AI will be integrated to explain the ML prediction,
# analyze suspicious elements, and answer user questions.

## Machine Learning Models

# 1. Logistic Regression
# 2. Multinomial Naive Bayes
# 3. Random Forest

In [3]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import joblib
import os
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv("../dataset/spam_email_dataset.csv")
df.head()

,email_id,subject,email_text,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_email,sender_domain,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
0,0,Weekly Report,budget review - Statement our I claim world st...,19,114,0,2,0,2,1,lctvdzm@outlook.com,outlook.com,0.66,19,3,0,23,0,0,0
1,1,Project Update,team sync - President series today already. In...,18,114,0,7,0,0,0,pxyldmi@company.com,company.com,0.95,4,4,0,16,1,0,0
2,2,🔥WIN BIG NOW!!,win free urgent offer limited limited urgent u...,19,126,0,4,1,1,1,atvanls@unknownmail.cc,unknownmail.cc,0.68,3,0,0,10,1,1,1
3,3,🔥WIN BIG NOW!!,guarantee click now cash offer click now guara...,16,101,0,7,1,1,1,qalxcnf@chealdealz.xyz,chealdealz.xyz,0.69,19,5,1,25,1,1,1
4,4,Meeting Reminder,team sync - Significant property hotel not add...,18,111,0,7,1,2,1,xoiccxl@yahoo.com,yahoo.com,0.67,4,5,1,8,0,0,0


In [5]:
import random
import pandas as pd

# 1. Defined vocabulary components for dynamic generation
spam_subjects = [
    "Urgent: Verify your account immediately", "You won a free gift card", 
    "Exclusive crypto discount inside", "Claim your bitcoin reward now", 
    "Security alert: Unrecognized login attempt", "Double your income working from home",
    "Final notice: Payment overdue", "Congratulations! You are selected for $1000 cash",
    "Password reset requested for your bank", "Limited time offer: 90% off all items"
]

spam_bodies = [
    "Click this link to verify your password and claim your cash reward immediately.",
    "Limited time offer! Buy now and get massive discounts. Visit our secure portal.",
    "Your crypto wallet balance increased. Click here to withdraw your funds now.",
    "Your account will be suspended within 24 hours unless you verify credentials.",
    "Earn guaranteed daily income with zero risk. Register using the link below.",
    "We detected suspicious activity on your credit card. Update your billing details."
]

ham_subjects = [
    "Project status update for next week", "Notes from today's team call",
    "Quarterly performance report attached", "Quick question regarding the assignment",
    "Lunch plans for tomorrow afternoon", "Schedule updates for upcoming exam",
    "Code review request for model pipeline", "Submission confirmation for assignment",
    "Meeting agenda and slide deck", "Updated project documentation"
]

ham_bodies = [
    "Hi team, please find attached the slide deck for tomorrow's presentation.",
    "Hey, can you review my pull request when you get a chance today? Thanks!",
    "The team meeting has been rescheduled to 3 PM in conference room B.",
    "Please send over the updated dataset so we can finalize model training.",
    "Thanks for sharing the notes. I will review them and get back to you soon.",
    "Attached is the draft report. Let me know if any changes are required."
]

# 2. Build 250 Spam and 250 Ham samples (500 total)
random.seed(42)

synthetic_spam = [
    {
        'subject': f"{random.choice(spam_subjects)} (Ref: {random.randint(1000, 9999)})",
        'email_text': f"{random.choice(spam_bodies)} Promo code: {random.randint(100, 999)}.",
        'label': 1
    }
    for _ in range(250)
]

synthetic_ham = [
    {
        'subject': f"{random.choice(ham_subjects)} - {random.choice(['FYI', 'Draft', 'Final', 'Action Required'])}",
        'email_text': f"{random.choice(ham_bodies)} Regards, {random.choice(['Alex', 'Sam', 'Priya', 'Jordan'])}.",
        'label': 0
    }
    for _ in range(250)
]

# 3. Create DataFrame and shuffle
synthetic_df = pd.DataFrame(synthetic_spam + synthetic_ham)
synthetic_df = synthetic_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Append to your existing dataframe and re-process
df = pd.concat([df, synthetic_df], ignore_index=True)
df['full_text'] = df['subject'].fillna('') + " " + df['email_text'].fillna('')

print(f"Total rows in dataset: {len(df)}")
print(df['label'].value_counts())

Total rows in dataset: 10500
label
0    6255
1    4245
Name: count, dtype: int64


In [6]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])
df.columns.tolist()
df.info()
df.isnull().sum()
df.duplicated().sum()
df.describe()

Number of Rows: 10500
Number of Columns: 21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10500 entries, 0 to 10499
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   email_id                 10000 non-null  float64
 1   subject                  10500 non-null  object 
 2   email_text               10500 non-null  object 
 3   num_words                10000 non-null  float64
 4   num_characters           10000 non-null  float64
 5   num_exclamation_marks    10000 non-null  float64
 6   num_links                10000 non-null  float64
 7   has_suspicious_link      10000 non-null  float64
 8   num_attachments          10000 non-null  float64
 9   has_attachment           10000 non-null  float64
 10  sender_email             10000 non-null  object 
 11  sender_domain            10000 non-null  object 
 12  sender_reputation_score  10000 non-null  float64
 13  email_hour               10000 n

,email_id,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
count,10000.00000,10000.000000,10000.000000,10000.0,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10500.000000
mean,4999.50000,18.097000,115.072400,0.0,3.976400,0.466700,0.803600,0.599600,0.681972,11.589600,3.00580,0.287100,15.501100,0.398500,0.499300,0.404286
std,2886.89568,5.163764,33.781247,0.0,2.580447,0.498915,0.752252,0.490004,0.172825,6.897023,2.00309,0.452431,8.650577,0.489614,0.500025,0.490777
min,0.00000,5.000000,25.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.300000,0.000000,0.00000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,2499.75000,14.000000,91.000000,0.0,2.000000,0.000000,0.000000,0.000000,0.560000,6.000000,1.00000,0.000000,8.000000,0.000000,0.000000,0.000000
50%,4999.50000,18.000000,115.500000,0.0,4.000000,0.000000,1.000000,1.000000,0.690000,12.000000,3.00000,0.000000,16.000000,0.000000,0.000000,0.000000
75%,7499.25000,22.000000,139.000000,0.0,6.000000,1.000000,1.000000,1.000000,0.810000,18.000000,5.00000,1.000000,23.000000,1.000000,1.000000,1.000000
max,9999.00000,33.000000,221.000000,0.0,8.000000,1.000000,2.000000,1.000000,1.000000,23.000000,6.00000,1.000000,30.000000,1.000000,1.000000,1.000000


In [7]:
print(df["subject"].head())
print(df["email_text"].head())

0       Weekly Report
1      Project Update
2      🔥WIN BIG NOW!!
3      🔥WIN BIG NOW!!
4    Meeting Reminder
Name: subject, dtype: object
0    budget review - Statement our I claim world st...
1    team sync - President series today already. In...
2    win free urgent offer limited limited urgent u...
3    guarantee click now cash offer click now guara...
4    team sync - Significant property hotel not add...
Name: email_text, dtype: object


In [8]:
df["subject"] = df["subject"].fillna("")
df["email_text"] = df["email_text"].fillna("")

In [9]:
df["subject"] = df["subject"].astype(str)
df["email_text"] = df["email_text"].astype(str)

In [10]:
df["combined_text"] = (
    df["subject"].fillna("") + " " +
    df["email_text"].fillna("")
)


In [11]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_text"] = df["combined_text"].apply(clean_text)
print(df['clean_text'].head())

0    weekly report budget review statement our i cl...
1    project update team sync president series toda...
2    win big now win free urgent offer limited limi...
3    win big now guarantee click now cash offer cli...
4    meeting reminder team sync significant propert...
Name: clean_text, dtype: object


In [12]:
X = df["clean_text"]
Y = df["label"]

X_train,X_test,y_train,y_test=train_test_split(
    X,Y,test_size=0.2,random_state=42,stratify=df['label']
)

In [13]:
df["label"].value_counts()

label
0    6255
1    4245
Name: count, dtype: int64

In [14]:

vectorizer = TfidfVectorizer(ngram_range=(1, 2),max_features=10000)

# X_text = vectorizer.transform(df['combined_text'])
X_train_vec = vectorizer.fit_transform(X_train)  # Learn & transform
X_test_vec = vectorizer.transform(X_test)

os.makedirs("../models", exist_ok=True)

joblib.dump(
    vectorizer,
    "../models/tfidf_vectorizer.pkl"
)
print("TF-IDF vectorizer saved successfully!")

TF-IDF vectorizer saved successfully!


In [15]:

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

prediction1 = model.predict(X_test_vec)

print("Accuracy of LogisticRegression :",accuracy_score(y_test,prediction1))


nb_model = MultinomialNB()

nb_model.fit(X_train_vec, y_train)
prediction2 = nb_model.predict(X_test_vec)
print("Accuracy of MultinomialNB :",accuracy_score(y_test,prediction2))


rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_vec, y_train)
prediction3 = rf_model.predict(X_test_vec)
print("Accuracy of RandomForestClassifier :",accuracy_score(y_test,prediction3))

Accuracy of LogisticRegression : 1.0
Accuracy of MultinomialNB : 1.0
Accuracy of RandomForestClassifier : 1.0


In [16]:

# Predict on test data
y_pred = model.predict(X_test_vec)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Evaluation")
print("---------------------------")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Model Evaluation
---------------------------
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1 Score  : 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1251
           1       1.00      1.00      1.00       849

    accuracy                           1.00      2100
   macro avg       1.00      1.00      1.00      2100
weighted avg       1.00      1.00      1.00      2100



In [17]:
os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/spam_model.pkl")

print("ML model saved successfully!")

loaded_model = joblib.load("../models/spam_model.pkl")

loaded_vectorizer = joblib.load(
    "../models/tfidf_vectorizer.pkl"
)

print("Model loaded successfully!")
print("Vectorizer loaded successfully!")

ML model saved successfully!
Model loaded successfully!
Vectorizer loaded successfully!


In [18]:
import re

def predict_spam(email_text):

    # Convert email text into TF-IDF
    email_vector = loaded_vectorizer.transform([email_text])
    # print(email_vector)
    # Predict
    prediction = loaded_model.predict(email_vector)[0]

    # Probability
    probability = loaded_model.predict_proba(email_vector)[0]

    # Get spam probability
    spam_probability = probability[1]

    if prediction == 1:
        result = "SPAM"
    else:
        result = "NOT SPAM"

    return result, spam_probability


# email ="""
# Subject: Your order #49281 has been shipped

# Hi Alex,

# Good news! Your recent order #49281 containing 2 items is on its way.
# Tracking Number: FEDEX-9482910382

# You can view your delivery status anytime on your account dashboard.

# Thank you for shopping with us!
# Customer Support
# """

email = """
Hi team,

Our project meeting is scheduled for tomorrow
at 10 AM in the conference room.

Regards,
John
"""



def clean_text(email):
    email = str(email).lower()
    email = re.sub(r"http\S+|www\S+", " ", email)
    email = re.sub(r"[^a-zA-Z0-9\s]", " ", email)
    email = re.sub(r"\s+", " ", email).strip()
    return email


email_text = clean_text(email)

result, confidence = predict_spam(email_text)

print("Prediction:", result)
print("Spam Probability:", round(confidence * 100, 2), "%")

Prediction: NOT SPAM
Spam Probability: 1.83 %


In [19]:
test_emails = [
    "free money",
    "you won a prize",
    "claim your lottery reward",
    "urgent click this link",
    "free prize congratulations",
    "win cash now",
    "meeting tomorrow at 10 AM",
    "please send the project report"
]

for email in test_emails:

    cleaned = clean_text(email)

    vector = loaded_vectorizer.transform([cleaned])

    prediction = loaded_model.predict(vector)[0]
    probability = loaded_model.predict_proba(vector)[0]

    print("\nEmail:", email)
    print("Prediction:", prediction)
    print("Probability:", probability)


Email: free money
Prediction: 0
Probability: [0.66599679 0.33400321]

Email: you won a prize
Prediction: 0
Probability: [0.71069344 0.28930656]

Email: claim your lottery reward
Prediction: 0
Probability: [0.57510368 0.42489632]

Email: urgent click this link
Prediction: 1
Probability: [0.4236687 0.5763313]

Email: free prize congratulations
Prediction: 0
Probability: [0.56710492 0.43289508]

Email: win cash now
Prediction: 1
Probability: [0.01540537 0.98459463]

Email: meeting tomorrow at 10 AM
Prediction: 0
Probability: [0.97408042 0.02591958]

Email: please send the project report
Prediction: 0
Probability: [0.98479248 0.01520752]


In [22]:
# 1. Check total vocabulary size learned during training
vocab = vectorizer.vocabulary_
print("Total unique words learned:", len(vocab))

# 2. Check coverage on your real test input
test_input = "Urgent claim your free reward prize now  at  payment paY paid time all time rewards money movie move click more 500000"
test_words = test_input.lower().split()
recognized = [w for w in test_words if w in vocab]

print("Input words:", test_words)
print("Recognized by model:", recognized)
print(f"Vocabulary match rate: {(len(recognized) / len(test_words)) * 100:.1f}%")



Total unique words learned: 10000
Input words: ['urgent', 'claim', 'your', 'free', 'reward', 'prize', 'now', 'at', 'payment', 'pay', 'paid', 'time', 'all', 'time', 'rewards', 'money', 'movie', 'move', 'click', 'more', '500000']
Recognized by model: ['urgent', 'claim', 'your', 'free', 'reward', 'prize', 'now', 'at', 'payment', 'pay', 'time', 'all', 'time', 'money', 'movie', 'move', 'click', 'more']
Vocabulary match rate: 85.7%
